# HRA Portal — CDE, Bot Detection & Geography Analysis
**Author:** Pratham  
**Course:** E538/E438 — Information Visualization, Indiana University  
**Role:** Advanced Insights — Cell Data Explorer (CDE) + Bot vs. Human + Geography  

---

## Overview
This notebook provides deep-dive analysis on three advanced dimensions of HRA Portal usage:

- **CDE**: Trend analysis, feature distribution, bounce-rate investigation (Jan 2024 – Jan 2026)
- **Bot vs. Human**: Traffic classification breakdown and interface-level contamination estimate
- **Geography**: Global usage patterns, top countries, events-per-user anomaly detection

**Data Source:** Amazon CloudFront Logs (Oct 2023 – Mar 2026), processed via DuckDB on Parquet (Nachiket's pipeline)  
**Total Records in Dataset:** ~6.87M events across 186 countries

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import pycountry
import warnings
warnings.filterwarnings('ignore')

# ── Global Style (matches team palette) ──────────────────────────────────────
sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.titlesize': 15,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.frameon': False,
    'font.family': 'DejaVu Sans',
})

COLORS = {
    'RUI':   '#1f77b4',  # blue  (matches Shreyash)
    'EUI':   '#ff7f0e',  # orange
    'CDE':   '#2ca02c',  # green
    'OTHER': '#cccccc',
    'HUMAN': '#2ca02c',
    'BOT':   '#d62728',
    'AIBOT': '#ff7f0e',
}

print('Libraries loaded ✓')

In [ ]:
# ── Load CSVs from Nachiket's pipeline ───────────────────────────────────────
# Update paths if running locally
monthly = pd.read_csv('monthly_usage.csv')
feature = pd.read_csv('feature_frequency.csv')
traffic = pd.read_csv('traffic_summary.csv')
geo     = pd.read_csv('geography_summary.csv')

monthly['event_month'] = pd.to_datetime(monthly['event_month'])

# CDE subsets
cde_m = monthly[monthly['interface_guess'] == 'CDE'].sort_values('event_month').reset_index(drop=True)
cde_f = feature[feature['interface_guess'] == 'CDE'].copy()

print('Datasets loaded:')
print(f'  monthly_usage    : {monthly.shape}')
print(f'  feature_frequency: {feature.shape}')
print(f'  traffic_summary  : {traffic.shape}')
print(f'  geography_summary: {geo.shape}')
print(f'\nCDE monthly records: {len(cde_m)}')
print(f'CDE feature endpoints: {len(cde_f):,}')
print(f'\nTraffic breakdown:')
traffic['pct'] = (traffic['events'] / traffic['events'].sum() * 100).round(1)
print(traffic.to_string(index=False))

---
## 2. CDE — Cell Data Explorer Analysis

The CDE provides histogram and violin visualizations of cell-level HuBMAP datasets, along with data download and example-driven exploration.

### 2.1 CDE Monthly Usage Trend

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax1 = axes[0]
ax1.fill_between(cde_m['event_month'], cde_m['event_count'], alpha=0.15, color=COLORS['CDE'])
ax1.plot(cde_m['event_month'], cde_m['event_count'],
         color=COLORS['CDE'], linewidth=2.5, marker='o', markersize=5)

peak_idx = cde_m['event_count'].idxmax()
peak_row = cde_m.loc[peak_idx]
ax1.annotate(f"Spike: {int(peak_row['event_count']):,}\nOct 2024",
             xy=(peak_row['event_month'], peak_row['event_count']),
             xytext=(peak_row['event_month'] - pd.DateOffset(months=6), peak_row['event_count'] * 0.80),
             arrowprops=dict(arrowstyle='->', color='#333', lw=1.5),
             fontsize=10, color='#333',
             bbox=dict(boxstyle='round,pad=0.3', fc='#fff3cd', ec='#ffaa00', alpha=0.9))

# Trendline excluding the spike
cde_no_spike = cde_m[cde_m['event_count'] < 1000]
x_num = (cde_no_spike['event_month'] - cde_no_spike['event_month'].min()).dt.days
z = np.polyfit(x_num, cde_no_spike['event_count'], 1)
p = np.poly1d(z)
x_all = (cde_m['event_month'] - cde_no_spike['event_month'].min()).dt.days
ax1.plot(cde_m['event_month'], p(x_all), '--', color='#888', linewidth=1.5,
         label=f'Trend (excl. spike): {z[0]*30:+.1f} events/mo')
ax1.legend(fontsize=9)
ax1.set_title('CDE Monthly Event Count (Jan 2024 – Jan 2026)')
ax1.set_ylabel('Event Count')
ax1.set_xlabel('Month')
ax1.tick_params(axis='x', rotation=40)

ax2 = axes[1]
ax2.fill_between(cde_m['event_month'], cde_m['unique_users'], alpha=0.15, color='#7c3aed')
ax2.plot(cde_m['event_month'], cde_m['unique_users'],
         color='#7c3aed', linewidth=2.5, marker='s', markersize=5)
ax2.set_title('CDE Unique Users per Month (Jan 2024 – Jan 2026)')
ax2.set_ylabel('Unique Users')
ax2.set_xlabel('Month')
ax2.tick_params(axis='x', rotation=40)

fig.suptitle('Fig 1. CDE Usage Trend — Events & Unique Users', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig1_cde_trend.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** CDE had zero events until Jan 2024, indicating a late launch or API visibility date. The Oct 2024 spike (2,731 events, 772 users) is ~6–7x the surrounding baseline — consistent with a HuBMAP workshop or demo event. Critically, the spike generated **no sustained retention**: the following months revert to 290–625 events, and the trendline is nearly flat.

### 2.2 CDE Feature Endpoint Analysis

In [ ]:
cde_top = cde_f.sort_values('event_count', ascending=False).head(15).copy()
cde_top['label'] = cde_top['feature_proxy'].str.replace(r'^/ui--staging', '/staging', regex=True).str[:45]

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(cde_top['label'][::-1], cde_top['event_count'][::-1],
               color=COLORS['CDE'], edgecolor='white', linewidth=0.5)
bars[-1].set_color('#1a6b1a')

for bar, val in zip(bars, cde_top['event_count'][::-1]):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            f'{int(val):,}', va='center', fontsize=9)

ax.set_xlabel('Event Count')
ax.set_title('Fig 2. CDE — Top 15 Endpoint Usage (All Time)\nTop 2 entry points account for 78% of all CDE traffic',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_cde_feature_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
total_cde = cde_f['event_count'].sum()
print(f'Total CDE events: {total_cde:,}')
print(f'/cde/ share: {6276/total_cde*100:.1f}%')
print(f'/cde/ + /cde combined: {(6276+2045)/total_cde*100:.1f}%')

**Insight:** CDE exhibits a classic bounce-rate pattern. `/cde/` (entry page) = 59% of traffic; `/cde` (without slash) = 19%. Combined: **78% of CDE traffic never goes past the landing page**. Functional endpoints like `/cde/create` (334 events) and `/cde/visualize` (95 events) are barely used.

### 2.3 CDE Distribution — Histogram & Violin

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

normal_events = cde_m[cde_m['event_count'] < 1000]['event_count']
outlier_events = cde_m[cde_m['event_count'] >= 1000]['event_count']

ax1 = axes[0]
ax1.hist(normal_events, bins=12, color=COLORS['CDE'], alpha=0.8,
         edgecolor='white', linewidth=0.8, label='Normal months')
ax1.hist(outlier_events, bins=2, color='#d62728', alpha=0.9,
         edgecolor='white', linewidth=0.8, label='Outlier (Oct 2024 spike)')
ax1.axvline(normal_events.median(), color='#333', linestyle='--', linewidth=1.5,
            label=f'Median: {normal_events.median():.0f} events/mo')
ax1.set_xlabel('Monthly Event Count')
ax1.set_ylabel('Number of Months')
ax1.set_title('Distribution of CDE Monthly Events')
ax1.legend(fontsize=9)

ax2 = axes[1]
cde_m['year'] = cde_m['event_month'].dt.year.astype(str)
cde_violin = cde_m[cde_m['event_count'] < 1000].copy()
year_data = [cde_violin[cde_violin['year'] == str(y)]['event_count'].values for y in [2024, 2025]]

parts = ax2.violinplot(year_data, positions=[1, 2], widths=0.5, showmedians=True, showextrema=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(['#a8d5a2', '#2ca02c'][i])
    pc.set_alpha(0.75)
parts['cmedians'].set_color('#1a1a2e')
parts['cmedians'].set_linewidth(2)

for i, (data, pos) in enumerate(zip(year_data, [1, 2])):
    jitter = np.random.normal(0, 0.04, size=len(data))
    ax2.scatter(pos + jitter, data, alpha=0.7, s=40, zorder=3, color=['#1a6b1a', '#0d3d0d'][i])

ax2.set_xticks([1, 2])
ax2.set_xticklabels(['2024\n(excl. Oct spike)', '2025'])
ax2.set_ylabel('Monthly Event Count')
ax2.set_title('CDE Monthly Events: 2024 vs 2025')

fig.suptitle('Fig 3. CDE — Statistical Distribution of Monthly Usage', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig3_cde_histogram_violin.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Bot vs. Human Traffic Analysis

In [ ]:
total_events = traffic['events'].sum()
traffic['pct'] = traffic['events'] / total_events * 100

labels = ['Likely Human', 'Bot', 'AI-Assistant / Bot']
sizes  = traffic['events'].values
colors_pie = [COLORS['HUMAN'], COLORS['BOT'], COLORS['AIBOT']]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax1 = axes[0]
wedges, texts, autotexts = ax1.pie(
    sizes, autopct='%1.1f%%', startangle=90, colors=colors_pie, explode=(0.04,0.04,0.04),
    pctdistance=0.78, wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2))
for at in autotexts:
    at.set_fontsize(12); at.set_fontweight('bold'); at.set_color('white')
ax1.legend(wedges, [f'{l}\n({e:,} events)' for l, e in zip(labels, sizes)],
           loc='lower center', bbox_to_anchor=(0.5, -0.15), fontsize=10, frameon=False)
ax1.set_title('Traffic Composition\nAll 6.87M Events', fontsize=13, fontweight='bold')
ax1.text(0, 0, f'6.87M\nTotal', ha='center', va='center', fontsize=12, fontweight='bold')

ax2 = axes[1]
x = np.arange(3); w = 0.35
event_pcts = traffic['pct'].values
user_pcts  = traffic['users'] / traffic['users'].sum() * 100
b1 = ax2.bar(x - w/2, event_pcts, w, color=colors_pie, alpha=0.9, edgecolor='white', label='% of Events')
b2 = ax2.bar(x + w/2, user_pcts,  w, color=colors_pie, alpha=0.5, edgecolor='white', hatch='//', label='% of Users')
for bar in b1:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{bar.get_height():.1f}%', ha='center', fontsize=9, fontweight='bold')
for bar in b2:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{bar.get_height():.1f}%', ha='center', fontsize=9, color='#555')
ax2.set_xticks(x); ax2.set_xticklabels(labels, fontsize=10)
ax2.set_ylabel('Percentage of Total')
ax2.set_title('Events vs Users by Traffic Type', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10); ax2.set_ylim(0, 80)

fig.suptitle('Fig 5. Bot vs Human Traffic Analysis — HRA Portal', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig5_bot_vs_human.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Human events: {traffic[traffic['traffic_type']=='Likely Human']['events'].values[0]:,} ({traffic[traffic['traffic_type']=='Likely Human']['pct'].values[0]:.1f}%)")
print(f"All bot events: {traffic[traffic['traffic_type'].str.contains('Bot')]['events'].sum():,} ({traffic[traffic['traffic_type'].str.contains('Bot')]['pct'].sum():.1f}%)")

**Insight:** 37.3% of all HRA events are non-human. All interface event totals from Shreyash's analysis (RUI: 818K, EUI: 607K) include this overhead — human-only estimated totals are RUI ≈ 513K, EUI ≈ 380K, CDE ≈ 6,660.

---
## 4. Geography Analysis

In [ ]:
def iso2_to_iso3(code):
    try:
        result = pycountry.countries.get(alpha_2=code)
        return result.alpha_3 if result else None
    except: return None

def iso2_to_name(code):
    try:
        result = pycountry.countries.get(alpha_2=code)
        return result.name if result else code
    except: return code

geo['iso3'] = geo['c_country'].apply(iso2_to_iso3)
geo['country_name'] = geo['c_country'].apply(iso2_to_name)
geo_valid = geo.dropna(subset=['iso3']).copy()

fig_map = px.choropleth(
    geo_valid, locations='iso3', color='events',
    hover_name='country_name',
    hover_data={'iso3': False, 'events': ':,', 'users': ':,'},
    color_continuous_scale=[[0,'#f7fbff'],[0.05,'#c6dbef'],[0.15,'#6baed6'],[0.4,'#2171b5'],[0.7,'#08306b'],[1,'#03162b']],
    title='Fig 7. HRA Portal — Global Usage by Country (Total Events)',
    labels={'events': 'Total Events'},
)
fig_map.update_layout(
    geo=dict(showframe=False, showcoastlines=True, coastlinecolor='#ccc',
             bgcolor='#fafcff', projection_type='natural earth'),
    coloraxis_colorbar=dict(title='Events', tickformat=','),
    margin=dict(l=0, r=0, t=50, b=0), paper_bgcolor='#fafcff',
)
fig_map.write_html('fig7_global_map.html')
fig_map.show()
print('Interactive map saved to fig7_global_map.html')

In [ ]:
geo_top20 = geo_valid.sort_values('events', ascending=False).head(20).copy()
geo_top20['events_per_user'] = (geo_top20['events'] / geo_top20['users']).round(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
top15 = geo_valid.sort_values('events', ascending=False).head(15)
bar_colors = ['#08306b' if c=='US' else '#6baed6' for c in top15['c_country']]

ax1 = axes[0]
bars = ax1.barh(top15['country_name'][::-1], top15['events'][::-1], color=list(reversed(bar_colors)), edgecolor='white')
for bar, val in zip(bars, top15['events'][::-1]):
    ax1.text(bar.get_width()+5000, bar.get_y()+bar.get_height()/2, f'{int(val):,}', va='center', fontsize=8.5)
ax1.set_title('Top 15 Countries by Events', fontsize=12, fontweight='bold')
ax1.set_xlabel('Total Events')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x/1000)}K'))

ax2 = axes[1]
geo_ratio = geo_top20.sort_values('events_per_user', ascending=False)
ratio_colors = ['#d62728' if r>20 else '#ff7f0e' if r>10 else '#2ca02c' for r in geo_ratio['events_per_user']]
bars2 = ax2.bar(geo_ratio['c_country'], geo_ratio['events_per_user'], color=ratio_colors, edgecolor='white')
ax2.axhline(geo_ratio['events_per_user'].median(), color='#333', linestyle='--', linewidth=1.5)
ax2.set_title('Events/User Ratio — Bot Suspicion Index\n(Red >20, Orange 10–20, Green <10)',
              fontsize=12, fontweight='bold')
ax2.set_xlabel('Country')
ax2.set_ylabel('Events per Unique User')
ax2.tick_params(axis='x', rotation=45)

fig.suptitle('Fig 8–9. Geographic Usage Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig8_top_countries.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 countries events/user:')
print(geo_ratio[['c_country','country_name','events','users','events_per_user']].head(10).to_string(index=False))

---
## 5. Summary of Key Findings

### CDE Findings
1. **Total events (Jan 2024 – Jan 2026): 10,624** — the smallest interface by far.
2. **Median monthly events (excl. Oct 2024 spike): 359** — essentially flat for 22 months.
3. **Oct 2024 spike (2,731 events, 772 users)** — a one-time event-driven surge with zero sustained retention.
4. **78% of CDE traffic is entry-point bounce** — users reach `/cde/` and do not proceed to functional pages.
5. **Power-law endpoint distribution** — top 2 endpoints = 78% of all CDE traffic; remaining 136 endpoints are minimal.
6. **Weak but real user growth**: median unique users improved from ~175/month (2024) to ~240/month (2025).

### Bot Detection Findings
1. **Likely Human: 4,305,153 events (62.7%)** — the genuine research audience.
2. **Bot: 2,236,420 events (32.5%)** — traditional crawlers and scrapers.
3. **AI-Assistant / Bot: 329,004 events (4.8%)** — LLM training crawlers (GPTBot, ClaudeBot, etc.).
4. **All interface totals are inflated ~37%** — bot-adjusted estimates: RUI ≈ 513K, EUI ≈ 380K, CDE ≈ 6,660 human events.
5. **Bots generate disproportionate events**: 37.3% of events but only 19.6% of unique IPs.

### Geography Findings
1. **186 countries reached** — strong global scientific audience.
2. **US: 74.7% of all events** — consistent with NIH-funded platform.
3. **Singapore #2** with ~273K events and relatively normal events/user ratio (~9.9) — likely a legitimate Asia-Pacific hub.
4. **HK events/user ratio: 52.4** — anomalously high; likely proxy/bot node.
5. **US events/user: 54.4** — highest ratio of all countries, confirming most bot traffic originates from US-based crawler infrastructure.
6. **Brazil events/user: 2.6** — cleanest non-US traffic signal.